In [1]:
# Spark Session to read and print employees_records.csv
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EmployeeRecordsSession") \
    .getOrCreate()

# sc = spark.sparkContext

orders = spark.read.csv("orders_dataset.csv", header=True, inferSchema=True)
customers = spark.read.csv("customers_dataset.csv", header=True, inferSchema=True)
products = spark.read.csv("products_dataset.csv", header=True, inferSchema=True)
orders.show(10)
# Create 3 temporary tables/views
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")
products.createOrReplaceTempView("products")

+--------+-----------+----------+----------+----------+--------+----------+-------------------+------------+------------+------------------+
|order_id|customer_id|product_id|order_date| ship_date|quantity|unit_price|discount_percentage|payment_mode|order_status|      sales_amount|
+--------+-----------+----------+----------+----------+--------+----------+-------------------+------------+------------+------------------+
|       1|         45|        27|2025-06-16|2025-06-22|       4|      6479|                 15|  Debit Card|   Delivered|           22028.6|
|       2|         64|        19|2026-02-14|2026-02-21|       3|     27477|                 22|         UPI|     Shipped|          64296.18|
|       3|         15|        50|2026-03-23|2026-03-30|       3|     17041|                 10|         UPI|     Shipped|46010.700000000004|
|       4|         47|        40|2025-10-08|2025-10-16|       1|     12793|                 16| Net Banking|   Delivered|10746.119999999999|
|       5|   

In [8]:
spark.sql("""
SELECT
    DATE_FORMAT(order_date, 'EEEE') AS weekday,
    SUM(sales_amount) AS total_revenue

FROM orders

GROUP BY DATE_FORMAT(order_date, 'EEEE')

ORDER BY total_revenue DESC
""").show()

+---------+-----------------+
|  weekday|    total_revenue|
+---------+-----------------+
|   Monday|5655873.800000001|
| Saturday|5031840.670000001|
|Wednesday|       4647350.79|
|  Tuesday|       4534374.49|
|   Sunday|4483950.159999999|
| Thursday|4423944.610000001|
|   Friday|       4280333.46|
+---------+-----------------+



In [5]:
#13. 7-Day Moving Average Sales
spark.sql("""
WITH daily_sales AS (

    SELECT
        order_date,
        SUM(sales_amount) AS daily_revenue

    FROM orders

    GROUP BY order_date
)

SELECT
    order_date,
    daily_revenue,

    AVG(daily_revenue) OVER (
        ORDER BY order_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS seven_day_moving_average

FROM daily_sales

ORDER BY order_date
""").show()

+----------+------------------+------------------------+
|order_date|     daily_revenue|seven_day_moving_average|
+----------+------------------+------------------------+
|2025-05-07|179179.03999999998|      179179.03999999998|
|2025-05-08|          81359.38|      130269.20999999999|
|2025-05-09| 75555.23999999999|      112031.21999999999|
|2025-05-10|         490804.75|      206724.60249999998|
|2025-05-12|329271.19000000006|               231233.92|
|2025-05-14|          70558.95|      204454.75833333333|
|2025-05-16|10617.619999999999|       176763.7385714286|
|2025-05-17| 59956.92999999999|      159732.00857142857|
|2025-05-19|         254054.68|       184402.7657142857|
|2025-05-21| 76966.06000000001|       184604.3114285714|
|2025-05-22|         162546.86|      137710.32714285716|
|2025-05-23|          54300.09|       98428.74142857142|
|2025-05-24|465520.32999999996|       154851.7957142857|
|2025-05-25|262350.39999999997|       190813.6214285714|
|2025-05-26|          114203.0|

In [6]:
#14. Revenue Contribution Percentage
spark.sql("""
WITH customer_revenue AS (

    SELECT
        customer_id,
        SUM(sales_amount) AS customer_revenue

    FROM orders

    GROUP BY customer_id
),

total_revenue AS (

    SELECT
        SUM(sales_amount) AS total_revenue

    FROM orders
)

SELECT
    cr.customer_id,
    cr.customer_revenue,

    ROUND(
        (cr.customer_revenue / tr.total_revenue) * 100,
        2
    ) AS revenue_contribution_percentage

FROM customer_revenue cr
CROSS JOIN total_revenue tr

ORDER BY revenue_contribution_percentage DESC
""").show()

+-----------+------------------+-------------------------------+
|customer_id|  customer_revenue|revenue_contribution_percentage|
+-----------+------------------+-------------------------------+
|         93|         845556.64|                           2.56|
|          6|         720304.25|                           2.18|
|         45|         591651.49|                           1.79|
|         11| 592001.0900000001|                           1.79|
|         80|         588679.86|                           1.78|
|        106|          581160.2|                           1.76|
|         52|         565407.64|                           1.71|
|         56|         532683.28|                           1.61|
|         64|         503299.37|                           1.52|
|         78| 494396.5199999999|                            1.5|
|        104|         497268.72|                            1.5|
|         94| 493296.8900000001|                           1.49|
|         48|         493

In [9]:
#15. Detect Fraudulent Orders
spark.sql("""
WITH order_history AS (

    SELECT
        order_id,
        customer_id,
        order_date,
        sales_amount,

        LAG(order_date) OVER (
            PARTITION BY customer_id
            ORDER BY order_date
        ) AS previous_order_time,

        LAG(sales_amount) OVER (
            PARTITION BY customer_id
            ORDER BY order_date
        ) AS previous_order_amount

    FROM orders
)

SELECT
    order_id,
    customer_id,
    order_date,
    sales_amount,
    previous_order_time,
    previous_order_amount,

    TIMESTAMPDIFF(
        MINUTE,
        previous_order_time,
        order_date
    ) AS minutes_since_previous_order

FROM order_history

WHERE
    TIMESTAMPDIFF(
        MINUTE,
        previous_order_time,
        order_date
    ) <= 5

    AND sales_amount > 10000

ORDER BY customer_id, order_date
""").show()

+--------+-----------+----------+------------+-------------------+---------------------+----------------------------+
|order_id|customer_id|order_date|sales_amount|previous_order_time|previous_order_amount|minutes_since_previous_order|
+--------+-----------+----------+------------+-------------------+---------------------+----------------------------+
|     421|         36|2025-08-29|     43974.9|         2025-08-29|    7667.400000000001|                           0|
|     430|         54|2026-01-30|    17241.36|         2026-01-30|              26462.7|                           0|
|     288|         78|2025-06-04|    35189.28|         2025-06-04|   171817.19999999998|                           0|
+--------+-----------+----------+------------+-------------------+---------------------+----------------------------+



In [10]:
spark.sql("""
WITH last_orders AS (

    SELECT
        customer_id,
        MAX(order_date) AS last_order_date
    FROM orders
    GROUP BY customer_id
),

latest_date AS (

    SELECT
        MAX(order_date) AS max_order_date
    FROM orders
)

SELECT
    c.customer_id,
    l.last_order_date,

    CASE
        WHEN l.last_order_date IS NULL
            THEN 'Never Ordered'

        WHEN DATEDIFF(
            d.max_order_date,
            l.last_order_date
        ) > 90
            THEN 'Inactive > 90 Days'
    END AS churn_status,

    CASE
        WHEN l.last_order_date IS NULL
            THEN NULL
        ELSE DATEDIFF(
            d.max_order_date,
            l.last_order_date
        )
    END AS inactive_days

FROM customers c

LEFT JOIN last_orders l
    ON c.customer_id = l.customer_id

CROSS JOIN latest_date d

WHERE
    l.last_order_date IS NULL
    OR DATEDIFF(
        d.max_order_date,
        l.last_order_date
    ) > 90

ORDER BY inactive_days DESC
""").show()

+-----------+---------------+------------------+-------------+
|customer_id|last_order_date|      churn_status|inactive_days|
+-----------+---------------+------------------+-------------+
|         66|     2025-05-08|Inactive > 90 Days|          364|
|         18|     2025-06-27|Inactive > 90 Days|          314|
|          4|     2025-07-07|Inactive > 90 Days|          304|
|         42|     2025-08-23|Inactive > 90 Days|          257|
|         75|     2025-09-08|Inactive > 90 Days|          241|
|         99|     2025-09-15|Inactive > 90 Days|          234|
|         63|     2025-09-23|Inactive > 90 Days|          226|
|          5|     2025-10-05|Inactive > 90 Days|          214|
|          1|     2025-10-08|Inactive > 90 Days|          211|
|         47|     2025-10-08|Inactive > 90 Days|          211|
|         71|     2025-10-22|Inactive > 90 Days|          197|
|        112|     2025-10-26|Inactive > 90 Days|          193|
|         43|     2025-11-04|Inactive > 90 Days|       

In [12]:
spark.sql("""
SELECT
    p1.product_name AS product_1,
    p2.product_name AS product_2,
    COUNT(*) AS times_bought_together

FROM orders o1

JOIN orders o2
    ON o1.customer_id = o2.customer_id
    AND o1.product_id < o2.product_id

JOIN products p1
    ON o1.product_id = p1.product_id

JOIN products p2
    ON o2.product_id = p2.product_id

GROUP BY
    p1.product_name,
    p2.product_name

ORDER BY times_bought_together DESC
""").show()

+--------------------+---------------+---------------------+
|           product_1|      product_2|times_bought_together|
+--------------------+---------------+---------------------+
|        Door Product|  Plant Product|                    5|
|        Door Product|  Throw Product|                    4|
|        Girl Product| Return Product|                    4|
|       Place Product|Brother Product|                    4|
|       Place Product|   Girl Product|                    4|
|   Interview Product|    Car Product|                    4|
|        Cold Product|  Throw Product|                    4|
|        Girl Product| Answer Product|                    4|
|    Business Product| Worker Product|                    4|
|        Cold Product| Answer Product|                    4|
|        Door Product|    Car Product|                    4|
|     Section Product|   Wear Product|                    3|
|       Place Product|   Five Product|                    3|
|        Cold Product|  

In [13]:
#18. Peak Sales Hour Analysis
spark.sql("""
SELECT
    HOUR(order_date) AS sales_hour,
    COUNT(*) AS total_orders

FROM orders

GROUP BY HOUR(order_date)

ORDER BY total_orders DESC
""").show()

+----------+------------+
|sales_hour|total_orders|
+----------+------------+
|         0|         500|
+----------+------------+



In [14]:
#to find peack hour
spark.sql("""
SELECT
    HOUR(order_date) AS sales_hour,
    COUNT(*) AS total_orders

FROM orders

GROUP BY HOUR(order_date)

ORDER BY total_orders DESC

LIMIT 1
""").show()

+----------+------------+
|sales_hour|total_orders|
+----------+------------+
|         0|         500|
+----------+------------+



In [15]:
# 19. Inventory Risk Detection
spark.sql("""
WITH sales_rate AS (

    SELECT
        product_id,
        SUM(quantity) / COUNT(DISTINCT order_date) AS avg_daily_sales

    FROM orders

    GROUP BY product_id
)

SELECT
    p.product_id,
    p.product_name,
    p.stock_quantity,
    s.avg_daily_sales,

    p.stock_quantity / s.avg_daily_sales AS estimated_days_remaining

FROM products p

JOIN sales_rate s
    ON p.product_id = s.product_id

WHERE
    s.avg_daily_sales > 0

ORDER BY estimated_days_remaining
""").show()

+----------+---------------+--------------+------------------+------------------------+
|product_id|   product_name|stock_quantity|   avg_daily_sales|estimated_days_remaining|
+----------+---------------+--------------+------------------+------------------------+
|        45|Message Product|            11|2.6666666666666665|                   4.125|
|        57|   Wear Product|            20|3.1666666666666665|       6.315789473684211|
|        55|    Bad Product|            30|3.8333333333333335|       7.826086956521739|
|        36|Section Product|            37| 2.888888888888889|      12.807692307692308|
|        48|Million Product|            46|               3.0|      15.333333333333334|
|        51|Brother Product|            46|2.9166666666666665|      15.771428571428572|
|        30|  Throw Product|            37|             2.125|       17.41176470588235|
|        42|   Name Product|            60|             3.375|       17.77777777777778|
|         3|Capital Product|    

In [16]:
#20. Second Highest Revenue Customer Per State
spark.sql("""
WITH customer_revenue AS (

    SELECT
        c.state,
        c.customer_id,
        c.customer_name,
        SUM(o.sales_amount) AS total_revenue

    FROM customers c

    JOIN orders o
        ON c.customer_id = o.customer_id

    GROUP BY
        c.state,
        c.customer_id,
        c.customer_name
),

ranked_customers AS (

    SELECT
        *,
        DENSE_RANK() OVER (
            PARTITION BY state
            ORDER BY total_revenue DESC
        ) AS revenue_rank

    FROM customer_revenue
)

SELECT
    state,
    customer_id,
    customer_name,
    total_revenue

FROM ranked_customers

WHERE revenue_rank = 2

ORDER BY state
""").show()

+-----------+-----------+-----------------+------------------+
|      state|customer_id|    customer_name|     total_revenue|
+-----------+-----------+-----------------+------------------+
|      Delhi|        104|  Alison Mcdonald|         497268.72|
|  Karnataka|          6| Theresa Ferguson|         720304.25|
|Maharashtra|         85|    Courtney Bell|485519.66000000003|
| Tamil Nadu|         52|  Jennifer Hunter|         565407.64|
|  Telangana|         78|Joshua Washington| 494396.5199999999|
+-----------+-----------+-----------------+------------------+



In [17]:
#21. Consecutive Purchase Days — LAG / LEAD
spark.sql("""
WITH purchase_dates AS (

    SELECT DISTINCT
        customer_id,
        order_date

    FROM orders
),

purchase_history AS (

    SELECT
        customer_id,
        order_date,

        LAG(order_date) OVER (
            PARTITION BY customer_id
            ORDER BY order_date
        ) AS previous_purchase_date,

        LEAD(order_date) OVER (
            PARTITION BY customer_id
            ORDER BY order_date
        ) AS next_purchase_date

    FROM purchase_dates
)

SELECT
    customer_id,
    order_date,
    previous_purchase_date,
    next_purchase_date,

    DATEDIFF(
        order_date,
        previous_purchase_date
    ) AS days_from_previous,

    DATEDIFF(
        next_purchase_date,
        order_date
    ) AS days_to_next

FROM purchase_history

WHERE
    DATEDIFF(order_date, previous_purchase_date) = 1
    OR
    DATEDIFF(next_purchase_date, order_date) = 1

ORDER BY customer_id, order_date
""").show()

+-----------+----------+----------------------+------------------+------------------+------------+
|customer_id|order_date|previous_purchase_date|next_purchase_date|days_from_previous|days_to_next|
+-----------+----------+----------------------+------------------+------------------+------------+
|         10|2025-12-07|            2025-10-30|        2025-12-08|                38|           1|
|         10|2025-12-08|            2025-12-07|        2026-04-10|                 1|         123|
|         22|2026-03-05|            2025-08-13|        2026-03-06|               204|           1|
|         22|2026-03-06|            2026-03-05|              NULL|                 1|        NULL|
|         49|2025-07-15|                  NULL|        2025-07-16|              NULL|           1|
|         49|2025-07-16|            2025-07-15|        2026-02-07|                 1|         206|
|         52|2026-02-04|            2025-10-29|        2026-02-05|                98|           1|
|         

In [18]:
#22. Revenue Spike Detection
spark.sql("""
SELECT
    order_id,
    customer_id,
    order_date,
    sales_amount,

    AVG(sales_amount) OVER () AS average_sales

FROM orders

WHERE sales_amount > (
    SELECT AVG(sales_amount)
    FROM orders
)

ORDER BY sales_amount DESC
""").show()

+--------+-----------+----------+------------------+-----------------+
|order_id|customer_id|order_date|      sales_amount|    average_sales|
+--------+-----------+----------+------------------+-----------------+
|     298|         22|2025-08-13|          219585.6|116420.2396208531|
|     235|         62|2025-11-24|216510.19999999998|116420.2396208531|
|     442|         89|2026-01-06|209850.55000000002|116420.2396208531|
|     266|        111|2026-01-20|         203200.35|116420.2396208531|
|     418|         93|2025-05-12|201859.40000000002|116420.2396208531|
|     448|         63|2025-06-21|         199640.35|116420.2396208531|
|     199|        112|2025-10-26|         198064.35|116420.2396208531|
|     174|          2|2025-12-04|196283.40000000002|116420.2396208531|
|      61|        114|2025-11-22|          193912.6|116420.2396208531|
|     405|         16|2026-03-15|         192800.25|116420.2396208531|
|     224|         11|2026-02-01|          192706.8|116420.2396208531|
|     